# 10. Full Pipeline Example

This notebook demonstrates the complete MARES pipeline end-to-end. Since real LLM calls require API keys, we'll use mock agents to show the flow without dependencies.

In production, set `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`) in your `.env` file.

In [ ]:
# Check for API keys
import os

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
if api_key and api_key != "test-key":
    print("API key found — you can run with a real LLM!")
else:
    print("No real API key set. Using mock/demo mode (no actual LLM calls).")

## End-to-End with Mock Agents

In [ ]:
from unittest.mock import AsyncMock, MagicMock
import json

from mares.orchestrator.orchestrator import Orchestrator

# Create mock LLM factory
mock_factory = MagicMock()
mock_factory.generate = AsyncMock()

# Cycle through JSON responses for each agent
responses = [
    # Planner
    json.dumps({
        "tasks": [
            {"id": 1, "task": "Research task scheduling", "depends_on": []},
            {"id": 2, "task": "Analyze failure patterns", "depends_on": [1]},
            {"id": 3, "task": "Synthesize findings", "depends_on": [2]},
        ]
    }),
    # Research for task 1
    json.dumps({
        "summary": "Task scheduling in distributed systems uses queues.",
        "facts": ["Celery uses Redis/RabbitMQ"], "sources": []
    }),
    # Research for task 2
    json.dumps({
        "summary": "Failures often stem from worker crashes.",
        "facts": ["Prefetch can cause issues"], "sources": []
    }),
    # Research for task 3
    json.dumps({
        "summary": "Synthesis complete.", "facts": ["Key insight"],
        "sources": []
    }),
    # Critic
    json.dumps({
        "passed": True, "issues": [], "summary": "All outputs look good."
    }),
    # Synthesizer
    "# MARES Final Report\n\nAll tasks completed successfully."
]

mock_factory.generate.side_effect = responses

# Override agents with our mock
from mares.agents.planner_agent import PlannerAgent
from mares.agents.research_agent import ResearchAgent
from mares.agents.critic_agent import CriticAgent
from mares.agents.synthesizer_agent import SynthesizerAgent

# Run with mock
orch = Orchestrator(
    planner=PlannerAgent(llm_factory=mock_factory),
    research=ResearchAgent(llm_factory=mock_factory),
    critic=CriticAgent(llm_factory=mock_factory),
    synthesizer=SynthesizerAgent(llm_factory=mock_factory),
)

print("Pipeline ready. Run with: await orch.run('Your task here')")

## Running Real Examples

To run with real LLMs, use the provided example scripts:

In [ ]:
print("""
Example commands:

  # Basic task
  python -m examples.run_basic_task

  # Research deep-dive
  python -m examples.run_research_task

  # Data pipeline with code
  python -m examples.run_data_pipeline_task

  # Algorithm implementation
  python -m examples.run_algorithm_task

  # Celery failure analysis (canonical demo)
  python -m examples.run_celery_analysis

  # Start the API server
  uvicorn api.main:app --reload
""")

## Next Steps

1. Copy `.env.example` to `.env` and add your API keys
2. Run examples with `python -m examples.run_basic_task`
3. Start the API: `uvicorn api.main:app --reload`
4. Read the docs in the `docs/` directory
5. Explore the source code in `mares/`

Happy multi-agent orchestrating! 🚀